In [17]:
!pip -q install rdkit

import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import numpy as np


df = pd.read_csv("ecoli_dataset_cleaned.csv")
print(df.head())
print(df.shape)
print(df["label"].value_counts())


                                              SMILES  label
0  Cc1cc(O)c(C(=O)NC(C(=O)NC2C(=O)N3C(C(=O)O)=C(C...      1
1  CON=C1CN(c2nc3c(cc2F)c(=O)c(C(=O)O)cn3C2CC2)CC...      1
2  CCC(C)CCCCC(=O)NC(CCN)C(=O)NC(C(=O)NC(CCN)C(=O...      1
3                                   Cl.N=C(N)n1cccn1      1
4  Cl.Cl.N=C(NCCCCCCNC(=N)NC(=N)Nc1ccc(Cl)cc1)NC(...      1
(2335, 2)
label
0    2215
1     120
Name: count, dtype: int64


In [18]:

# Check validity
df["valid_smiles"] = df["SMILES"].apply(lambda s: isinstance(s, str) and s.strip() != "" and Chem.MolFromSmiles(s) is not None)

n_total = len(df)
n_invalid = (~df["valid_smiles"]).sum()

print(f"Total rows: {n_total}")
print(f"Invalid SMILES found: {n_invalid}")

# Show invalid rows (if any)
if n_invalid > 0:
    print("\nInvalid row(s):")
    print(df.loc[~df["valid_smiles"], ["SMILES", "label"]].head())

# Filter + Confirm
df = df.loc[df["valid_smiles"]].drop(columns=["valid_smiles"]).reset_index(drop=True)

print(f"\nAfter filtering rows: {len(df)}")
print("Dataset filtered. All SMILES are valid.")


Total rows: 2335
Invalid SMILES found: 1

Invalid row(s):
                                                 SMILES  label
2244  O=S(=O)(OCC1OC(OC2(COS(=O)(=O)O[AlH3](O)O)OC(C...      0

After filtering rows: 2334
Dataset filtered. All SMILES are valid.


[13:15:19] Explicit valence for atom # 16 Al, 6, is greater than permitted


In [21]:
#Convert SMILES to ECFP (Morgan)
from rdkit.Chem import rdFingerprintGenerator

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_ecfp_array(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = morgan_gen.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


# Build X and y
X_list = []
y_list = []

for smi, label in zip(df["SMILES"], df["label"]):
    arr = smiles_to_ecfp_array(smi)
    if arr is not None:
        X_list.append(arr)
        y_list.append(label)


X = np.vstack(X_list)           # shape: (n_samples, 2048)
y = np.array(y_list, dtype=int) # shape: (n_samples,)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positives %:", y.mean() * 100)


X shape: (2334, 2048)
y shape: (2334,)
Positives %: 5.141388174807198


In [22]:
#Save output files
np.save("X_ecfp2048.npy", X)
np.save("y.npy", y)

# keep a mapping back to SMILES
df.loc[:len(y)-1, ["SMILES", "label"]].to_csv("smiles_labels_valid.csv", index=False)


In [23]:
#Data check
#all-zero fingerprints
all_zero = (X.sum(axis=1) == 0).sum()
print("All-zero rows:", all_zero)

#Duplicate SMILES after filtering
print("Duplicate SMILES:", df["SMILES"].duplicated().sum())


All-zero rows: 0
Duplicate SMILES: 0
